# Notebook 12 — Mini Project and Module Assessment

**Module:** 02 — Mathematics for Biology  
**Notebook:** 12 of 12 — Module capstone  
**Prerequisites:** All NB01–NB11  
**Time estimate:** 150 minutes

---
## Notebook Purpose

This notebook has three parts:

1. **Integration audit** — verify that every deliverable from NB01–NB11 exists and runs
2. **Oral exam self-test** — structured, timed, no notes — for Track A interview readiness
3. **Mini project completion** — polish the Lotka-Volterra portfolio artifact and confirm
   the figure is saved in `portfolio/`

**Pass gate:** ≥ 80% on the oral exam self-assessment. Do not mark this module complete
until the gate is passed.

---
## Part 1 — Integration Audit

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp, quad
from scipy.optimize import curve_fit, brentq
from collections import Counter
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
import json

In [ ]:
# Cell 1.1 — Notebook existence check
nb_dir = Path(".").parent / "notebooks"
expected_notebooks = [f"{i:02d}_" for i in range(1, 13)]

print("=" * 60)
print("MODULE 02 — NOTEBOOK AUDIT")
print("=" * 60)

all_found = True
for prefix in expected_notebooks:
    found = list(nb_dir.glob(f"{prefix}*.ipynb"))
    status = "FOUND" if found else "MISSING"
    name = found[0].name if found else "—"
    if not found:
        all_found = False
    print(f"  [{status}]  {name}")

print()
print(f"All notebooks present: {'YES' if all_found else 'NO — complete missing notebooks first'}")

In [ ]:
# Cell 1.2 — Portfolio figure check
portfolio_path = Path("../../../../portfolio/module02_lotka_volterra_phase_portrait.png")
print(f"Portfolio figure exists: {'YES' if portfolio_path.exists() else 'NO — run NB09 Cell 7.1'}")
if portfolio_path.exists():
    size_kb = portfolio_path.stat().st_size / 1024
    print(f"File size: {size_kb:.1f} KB")

In [ ]:
# Cell 1.3 — Function inventory: key implementations from each notebook
REQUIRED = {
    "NB03": ["riemann_sum", "trapezoid_auc"],
    "NB04": ["exponential_growth", "logistic_growth"],
    "NB05": ["logistic_map", "bifurcation_diagram"],
    "NB07": ["euler"],
    "NB08": ["rk4"],
    "NB09": ["lotka_volterra"],
    "NB10": ["count_kmers"],
    "NB11": ["gradient_descent_1d", "numerical_gradient"],
}

print("Function inventory (self-reported — paste implementation below each to verify):")
for nb, fns in REQUIRED.items():
    print(f"  {nb}: {', '.join(fns)}")

print("\nAction: paste each function implementation in the cells below and verify it runs.")

In [ ]:
# Cell 1.4 — End-to-end pipeline: logistic growth → fit → AUC → phase portrait
# This cell exercises all major topics in one coherent pipeline.

rng = np.random.default_rng(0)

# 1. Generate logistic growth data
r_true, K_true, N0_true = 0.4, 500.0, 10.0
t_obs = np.linspace(0, 30, 20)
N_true_vals = K_true / (1 + (K_true/N0_true - 1) * np.exp(-r_true * t_obs))
N_obs = N_true_vals * rng.lognormal(0, 0.08, len(t_obs))

# 2. Fit logistic model
def logistic_growth(t, r, K, N0):
    return K / (1 + (K/N0 - 1) * np.exp(-r * t))

popt, _ = curve_fit(logistic_growth, t_obs, N_obs, p0=[0.3, 400, 8], bounds=([0,0,0],[2,1e4,100]))
r_fit, K_fit, N0_fit = popt

# 3. Compute AUC numerically (drug exposure analogy)
AUC_trap = np.trapz(N_obs, t_obs)

# 4. Solve Lotka-Volterra
def lv(t, y): return [1.0*y[0] - 0.1*y[0]*y[1], 0.075*y[0]*y[1] - 1.5*y[1]]
sol = solve_ivp(lv, (0, 30), [10, 5], t_eval=np.linspace(0, 30, 500), rtol=1e-8)

print("End-to-end pipeline complete.")
print(f"  Logistic fit: r={r_fit:.3f} (true {r_true}), K={K_fit:.1f} (true {K_true})")
print(f"  AUC (trapezoid): {AUC_trap:.1f}")
print(f"  Lotka-Volterra: prey range [{sol.y[0].min():.1f}, {sol.y[0].max():.1f}]")

---
## Part 2 — Oral Exam Self-Test

**Rules:** Set a timer. Answer each section without looking at your notes or notebooks.
Score yourself honestly using the rubric below. Record your score in the final cell.

**Time budget:** 45 minutes total.

### Section A — Calculus Fundamentals (10 points)

1. Write the central difference formula for a numerical derivative. What is its error order? (2 pts)
2. What is the Fundamental Theorem of Calculus in one sentence? (1 pt)
3. What does AUC mean in pharmacokinetics? Write the formula for $\text{AUC}_{0\to\infty}$
   for $C(t) = C_0 e^{-k_e t}$. (2 pts)
4. Solve $\int_0^\infty e^{-2t}\,dt$ analytically. (2 pts)
5. What is the optimal step size $h$ for the central difference, and why does a very
   small $h$ actually increase error? (3 pts)

### Section B — ODE Models (20 points)

1. Write the logistic ODE. Solve it analytically by separation of variables. (5 pts)
2. What are the fixed points of the logistic model? Which is stable? (3 pts)
3. Write the Lotka-Volterra system. Define every parameter biologically. (4 pts)
4. What is the coexistence fixed point of the Lotka-Volterra system? Derive it. (4 pts)
5. What is a conservation law? What does it imply about Lotka-Volterra trajectories? (4 pts)

### Section C — Numerical Methods (15 points)

1. Write the Euler update equation. What is its global error order? (3 pts)
2. Write all four RK4 slope estimates and the final update. (6 pts)
3. Why is RK4 preferred over Euler for the same computational budget? (3 pts)
4. What is an adaptive step size solver? What does `rtol=1e-6` mean in `solve_ivp`? (3 pts)

### Section D — Discrete Mathematics and Optimization (10 points)

1. How many 8-mers are possible over the DNA alphabet? (1 pt)
2. Write the gradient descent update rule. What happens if $\eta$ is too large? (3 pts)
3. What is the logistic map? For which values of $r$ does it exhibit chaos? (3 pts)
4. Define a graph $G = (V, E)$. What is the degree of a node? (3 pts)

In [ ]:
# Cell 2.1 — Score recording and visualisation
# Fill in your scores after the timed self-test
scores = {
    "A_calculus":       None,  # max 10
    "B_ode_models":     None,  # max 20
    "C_numerical":      None,  # max 15
    "D_discrete_opt":   None,  # max 10
}
maxima = {"A_calculus": 10, "B_ode_models": 20, "C_numerical": 15, "D_discrete_opt": 10}

if any(v is None for v in scores.values()):
    print("ASSESSMENT NOT COMPLETE — fill in scores above before running this cell.")
else:
    total = sum(scores.values())
    total_max = sum(maxima.values())
    pct = 100 * total / total_max

    print(f"{'Section':<25} {'Score':>8} {'Max':>6} {'%':>8}")
    print("-" * 52)
    for sec, score in scores.items():
        mx = maxima[sec]
        print(f"  {sec:<23} {score:>8} {mx:>6} {100*score/mx:>7.0f}%")
    print("-" * 52)
    print(f"  {'TOTAL':<23} {total:>8} {total_max:>6} {pct:>7.0f}%")
    print()
    if pct >= 80:
        print("PASS — Module 02 complete. Add to progress tracker and start Module 03.")
    else:
        print(f"NOT YET — {pct:.0f}% < 80% threshold. Revisit weak sections before Module 03.")

---
## Part 3 — Mini Project Completion Checklist

Work through this checklist. Mark each item `[x]` when done.

**Lotka-Volterra portfolio artifact (NB09):**
- [ ] Biological motivation written (can explain to non-biologist)
- [ ] ODE system written with all parameters defined
- [ ] Coexistence fixed point derived algebraically
- [ ] Conservation law verified numerically
- [ ] Both Euler (from scratch) and `solve_ivp` solutions compared
- [ ] Phase portrait figure saved to `portfolio/module02_lotka_volterra_phase_portrait.png`
- [ ] Parameter sensitivity (alpha doubled): period change predicted and verified

**Oral exam self-test:**
- [ ] Timed, no notes
- [ ] Score ≥ 80/55 = 80%

**Module completion gate:**
- [ ] All 12 notebooks executed top-to-bottom without errors
- [ ] Portfolio figure exists
- [ ] Oral exam passed
- [ ] Reflection written below

---
## Module Reflection

**Date completed:** ____________________  
**Oral exam score:** ______ / 55  

**What I can now do that I couldn't at the start of this module:**

> *[Write 2-3 sentences. Be specific: 'I can derive the Lotka-Volterra fixed points and interpret the phase portrait without notes.']*

**What is still fuzzy:**

> *[One honest sentence. This informs the next review cycle.]*

**Track A readiness check:**

> If handed the logistic ODE cold in an interview and asked 'explain this and solve it', could you? [ ] Yes [ ] Mostly [ ] Not yet

---

**Module 02 — MATHEMATICS FOR BIOLOGY — COMPLETE**

*Next module: `03_statistics_and_probability/`*